# Swiggy Instamart Customer Intelligence & Support Copilot —swiggy - Capstone Project
Welcome to the master Jupyter Notebook for the **Instamart Capstone Project**. This notebook breaks down the complete implementation across all four core parts:
* **Part A:** Data Foundation (Cleaning, SQLite storage, SQL queries & exploratory analysis)
* **Part B:** Customer Segments & Lapse Prediction (RFM, K-means clustering, pipeline preprocessing, and classification)
* **Part C:** Support Ticket Triage with Guardrails (Input checks, regex masking, LLM retries, validation & routing)
* **Part D:** Policy RAG Assistant & ReAct Refund Agent (Chunking, TF-IDF retrieval, tool-calling agents & checkpoints)

---
## Part A — Data Foundation: Clean, Store, Query

In [ ]:
# Part A: Setup and Audit
import pandas as pd
import sqlite3
import os

# Load raw datasets
customers_df = pd.read_csv('data/raw/customers.csv')
products_df = pd.read_csv('data/raw/products.csv')
orders_df = pd.read_csv('data/raw/orders.csv')
tickets_df = pd.read_csv('data/raw/tickets.csv', keep_default_na=False)

print('Customers shape:', customers_df.shape)
print('Orders shape:', orders_df.shape)
print('Tickets shape:', tickets_df.shape)

In [ ]:
# Part A: Cleaning Pipeline & SQLite Database Export
os.makedirs('data/clean', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('sql', exist_ok=True)

# 1. Clean Customers
customers_df['city'] = customers_df['city'].str.strip().str.title()
customers_df['gender'] = customers_df['gender'].str.lower().map({'male': 'male', 'female': 'female'}).fillna('unknown')
customers_df.loc[~customers_df['age'].between(10, 100, inclusive='both'), 'age'] = None
median_ages = customers_df.groupby('city')['age'].transform('median')
customers_df['age'] = customers_df['age'].fillna(median_ages)
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'])

# 2. Clean Orders
orders_df['order_ts'] = pd.to_datetime(orders_df['order_ts'])
orders_df = orders_df.drop_duplicates()
# Keep latest record per order_id
orders_df = orders_df.sort_values('order_ts').drop_duplicates(subset=['order_id'], keep='last')

# Numeric conversion for amounts & delivery times
orders_df['order_amount'] = pd.to_numeric(orders_df['order_amount'], errors='coerce')
merged_prod = orders_df.merge(products_df, on='product_id', how='left')
missing_amt = orders_df['order_amount'].isna()
orders_df.loc[missing_amt, 'order_amount'] = round(
    merged_prod.loc[missing_amt, 'price_inr'] * merged_prod.loc[missing_amt, 'quantity'] * (1 - merged_prod.loc[missing_amt, 'discount_pct'] / 100), 2
)

orders_df.loc[orders_df['delivery_minutes'] < 0, 'delivery_minutes'] = None
median_deliv = orders_df.merge(products_df, on='product_id').groupby('category')['delivery_minutes'].transform('median')
completed_mask = (orders_df['status'] == 'completed') & (orders_df['delivery_minutes'].isna())
orders_df.loc[completed_mask, 'delivery_minutes'] = median_deliv[completed_mask]

# Save clean CSVs
customers_df.to_csv('data/clean/customers.csv', index=False)
products_df.to_csv('data/clean/products.csv', index=False)
orders_df.to_csv('data/clean/orders.csv', index=False)

# Store into SQLite
conn = sqlite3.connect('data/instamart.db')
customers_df.to_sql('customers', conn, if_exists='replace', index=False)
products_df.to_sql('products', conn, if_exists='replace', index=False)
orders_df.to_sql('orders', conn, if_exists='replace', index=False)
conn.close()
print('Part A Database initialized successfully!')

---
## Part B — Customer Segments and Lapse Prediction

In [ ]:
# Part B: RFM Calculation and K-Means Clustering
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import joblib

snapshot = pd.to_datetime('2025-10-01 00:00:00')
comp_orders = orders_df[orders_df['status'] == 'completed']

rfm = comp_orders.groupby('customer_id').agg(
    recency=('order_ts', lambda x: (snapshot - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('order_amount', 'sum')
).reset_index()

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

# Fit KMeans with K=4
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm['segment_id'] = kmeans.fit_predict(rfm_scaled)

# Identify Premium segment (highest mean monetary)
seg_monetary = rfm.groupby('segment_id')['monetary'].mean()
premium_cluster = seg_monetary.idxmax()
rfm['is_premium'] = rfm['segment_id'] == premium_cluster
rfm['segment_name'] = rfm['segment_id'].map({0: 'Regular', 1: 'Loyalist', 2: 'High-Value Champions', 3: 'At-Risk'})

rfm.to_csv('outputs/customer_segments.csv', index=False)
print('Customer segments saved successfully!')

---
## Part C — Support Ticket Triage with Guardrails

In [ ]:
# Part C: Guardrails, Redaction & Mock LLM Integration setup
import re

def redact_text(text):
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[EMAIL]', text)
    text = re.sub(r'\b[6-9]\d{9}\b', '[PHONE]', text)
    text = re.sub(r'ORD-\d{6}', '[ORDER_ID]', text)
    return text

# Example check
sample_ticket = 'My order ORD-123456 failed. Call me at 9876543210 or email test@example.com.'
print('Original:', sample_ticket)
print('Redacted:', redact_text(sample_ticket))

---
## Part D — Policy RAG Assistant and Refund Agent

In [ ]:
# Part D: Document Chunking and TF-IDF Retrieval Setup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

chunks = []
policy_dir = 'data/policies'
if os.path.exists(policy_dir):
    for fn in os.listdir(policy_dir):
        if fn.endswith('.txt'):
            with open(os.path.join(policy_dir, fn), 'r', encoding='utf-8') as f:
                words = f.read().split()
                for i in range(0, len(words), 30):
                    chunk_words = words[i:i+40]
                    if not chunk_words:
                        break
                    chunks.append({'source': fn, 'text': ' '.join(chunk_words)})
                    if i + 40 >= len(words):
                        break
    print(f'Total chunks generated: {len(chunks)}')
else:
    print('Policy directory not found. Run generate_data.py first.')